# 11. Conclusiones

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/processed/panel_horario.parquet').exists())
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':.15})
panel = pq.read_table(ROOT/'data/processed/panel_horario.parquet', filters=[('inicio','<',pd.Timestamp('2022-01-01',tz='America/Bogota'))]).to_pandas()
assert panel.index.max() < pd.Timestamp('2022-01-01',tz='America/Bogota')
assert panel.index.is_monotonic_increasing and panel.index.is_unique
y = panel['y_demanda_real_kwh']
gwh = y / 1e6
registro = json.loads((ROOT/'data/processed/feature_registry.json').read_text(encoding='utf-8'))


In [2]:
resumen_eda={
    'horas_desarrollo':len(panel),
    'crecimiento_pct':float(100*(gwh.loc['2021'].mean()/gwh.loc['2000'].mean()-1)),
    'acf24':float(acf(gwh,nlags=168,fft=True)[24]) if 'acf' in globals() else float(gwh.autocorr(24)),
    'acf168':float(gwh.autocorr(168))}
crecimiento_eda=resumen_eda['crecimiento_pct']; acf_eda24=resumen_eda['acf24']; acf_eda168=resumen_eda['acf168']


Entre 2000 y 2021, la demanda media aumenta un 80.8 %. Las autocorrelaciones a 24 y 168 horas son 0.93 y 0.97. El crecimiento y la persistencia aconsejan comparar referencias estacionales y considerar el nivel histórico de cada ventana, manteniendo los parámetros de transformación dentro del entrenamiento.

## Consecuencias para la representación y la evaluación

| Hallazgo | Decisión metodológica | Riesgo que debe controlarse |
|---|---|---|
| Nivel cambiante y diferencias de dispersión | Normalización por nivel conocido de la ventana y reporte de errores por año. | Confundir cambio de escala con pérdida de capacidad predictiva. |
| Ciclos diarios, semanales y festivos | Historia de 168 horas y calendario conocido para las 24 horas de salida. | Mezclar sábados con domingos o trasladar observaciones del horizonte a las entradas. |
| Anomalías y valores originales de hidrología | Conservar datos, indicadores y trazabilidad; evaluar sensibilidad antes de corregir. | Eliminar eventos reales o interpretar una razón como porcentaje. |
| Asociaciones climáticas y desplazamiento distributivo | Contrastar extensiones mediante ablación y los mismos orígenes. | Confundir asociación contemporánea, tendencia común y señal disponible al origen. |
| Cobertura y léxico variables entre fuentes | Ajustar representaciones con entrenamiento y medir cobertura al proyectar. | Atribuir a eventos un cambio que proviene del archivo o de la composición editorial. |
| Errores temporalmente relacionados | Comparaciones pareadas por origen y remuestreo de días en bloques. | Inflar precisión contando horas y semillas como observaciones independientes. |

## Alcance y amenazas a la validez

La demanda es nacional, mientras el clima resume ocho puntos urbanos; esa diferencia espacial limita la atribución de asociaciones a usos concretos. Los pesos de población de 2018 son estáticos y proceden de una fuente secundaria. El reanálisis meteorológico caracteriza retrospectivamente el tiempo observado, sin reconstruir por sí solo lo disponible en operación.

Las series administrativas y los titulares pueden contener revisiones. La disponibilidad declarada de noticias no acredita una versión histórica exacta; el escenario estricto ofrece mayor trazabilidad con menor cobertura. La ausencia de titulares recuperados no debe convertirse en una señal de ausencia de eventos.

La validación 2018–2021 combina regímenes distintos y participa en selección. La selección retrospectiva limita la independencia de las comparaciones sobre prueba. El EDA caracteriza asociaciones y cobertura, sin establecer causalidad ni una ganancia atribuible al texto.

## Referencia con demanda y calendario

Entre las cuatro familias con demanda y calendario, la MLP presenta el menor MASE medio del periodo de prueba; ARX y CNN1D quedan próximos y LSTM obtiene mayor error. Las cuatro familias superan las referencias estacionales en el agregado. La evaluación conserva cinco semillas por configuración y separa variabilidad algorítmica de incertidumbre temporal. Las diferencias se describen junto con perfiles horarios, segmentos, curvas de aprendizaje y costo de ajuste, sin afirmar superioridad confirmatoria.

## Evaluación conjunta de arquitecturas

El benchmark de referencia incluye quince procedimientos aprendidos y 75 realizaciones. El menor MASE medio corresponde a TimesNet · N (0,6429). El MCS al 90 % retiene TimesNet · N. El conjunto está condicionado al periodo de prueba, a las cinco inicializaciones y al protocolo de remuestreo.

TaTS adaptado · N+O reduce el MAE 0,61 % frente a su control, sin diferencia temporal demostrada; Fusión tardía · N+O aumenta el MAE 4,65 % frente a su control, con diferencia temporal detectada; TGForecaster · N+O aumenta el MAE 12,02 % frente a su control, con diferencia temporal detectada. Las entradas y los presupuestos distintos limitan la atribución exclusiva a la arquitectura. La sensibilidad al contenido y la comparación entre procedimientos mantienen interpretaciones separadas.

## Memoria residual para el pronóstico de demanda

M4ₙ alcanza MASE 0,5770 en los 1.461 orígenes de 2022–2025, frente a 0,6429 de TimesNet, el menor error del Top 10. La reducción relativa es 10,25 %. El MCS del 90 % que incorpora dieciocho procedimientos retiene únicamente a M4ₙ. Las tres pruebas temporales respaldan diferencias favorables en las comparaciones presentadas, con control de multiplicidad y pérdidas individuales promediadas entre semillas.

La demanda histórica domina la importancia por sustitución. La jerarquía de sensibilidad del contexto es más estable que la identidad de los episodios recuperados. La compuerta y la atención permiten examinar cómo interviene la memoria, sin atribuir causalidad a las variables o vecinos. La comparación caracteriza los procedimientos conservados: la desigualdad de presupuestos, las cinco semillas de los comparadores y la selección retrospectiva delimitan su generalización. La {ref}`evaluación ampliada <evaluacion-m4-ampliada>` reúne métricas, intervalos, interpretabilidad y eficiencia.

## Robustez de las entradas

El diagnóstico de estrés de 96 fechas y cinco realizaciones por procedimiento cuantifica la respuesta a ruido, interrupciones, valores atípicos y pérdida documental en tres severidades. El panel no acredita una mejora superior al 3 % de M4 frente a TimesNet después del ajuste de Holm. Bajo las perturbaciones impuestas, la ventaja descriptiva de M4 frente a TimesNet se invierte en 6 de las nueve condiciones numéricas. Las interrupciones y los valores atípicos de mayor amplitud incrementan especialmente su error. Las curvas y los intervalos delimitan la estabilidad del pipeline; los procedimientos sin canal textual mantienen su salida al retirar documentos. La memoria utiliza el archivo histórico disponible mientras se perturba la consulta. El benchmark con datos originales constituye la evaluación principal; la frecuencia real de fallos no se deduce de los escenarios simulados. El {ref}`análisis de robustez <robustez-congelada>` desarrolla estos resultados y su alcance temporal.

## Margen empírico de mejora

El selector retrospectivo de la menor pérdida diaria entre 18 procedimientos obtiene MASE 0,4675. M4 registra γ = 0,1234, con IC95 % [0,1075; 0,1414], y cierra el 87,66 % de la brecha desde el ingenuo semanal. La diversidad de los mínimos diarios muestra complementariedad entre procedimientos. Esta referencia optimista delimita el margen del conjunto evaluado sin identificar el error irreducible ni garantizar que la selección pueda realizarse en tiempo real. El {ref}`análisis del techo empírico <techo-empirico>` conserva el cálculo y su incertidumbre temporal.

## Estabilidad predictiva y estructura residual de M4

La desviación media entre diez semillas es 0,0511 GWh, equivalente al 0,56 % del nivel pronosticado. La asociación diaria entre dispersión y error es positiva y de magnitud limitada (ρ = 0,227). Los desacuerdos se concentran en ciertas fechas y horizontes; la atención de recuerdos y la estabilidad de la salida representan propiedades distintas.

El sesgo global es -0,0075 GWh y se detecta autocorrelación en 22 de 24 horizontes tras Holm. La banda p10–p90 entre semillas contiene el 30,02 % de las observaciones y no sustituye un intervalo predictivo calibrado. La evaluación por festivos y niveles horarios complementa el promedio global. El {ref}`diagnóstico propio de M4 <diagnosticos-m4>` conserva las matrices, pruebas y segmentación comparada.

## Sensibilidad a la dependencia temporal

La superioridad de M4 se mantiene con bloques de 14, 28 y 56 días y covarianzas HAC de 13, 27 y 55 rezagos. Frente a TimesNet, la reducción del MASE es 10,25 %; el IC95 % con bloques de 56 días es [8,64 %; 11,94 %]. La mejora superior al 3 % conserva respaldo tras Holm sobre 306 hipótesis. La {ref}`sensibilidad temporal <sensibilidad-temporal-m4>` utiliza las pérdidas guardadas y complementa el diagnóstico de dependencia residual.

## Incertidumbre con calibración común

En los 1.096 orígenes de 2023–2025, M4 obtiene el menor WIS (0,05080 GWh). La reducción frente a TimesNet es 10,20 %, con respaldo de bootstrap temporal y DM–HLN después de Holm(34). La calibración móvil se inicia con errores disponibles de 2022 y conserva los pesos. Al nivel nominal del 90 %, las coberturas observadas de M4 son 88,91 % por hora y 87,86 % por trayectoria, con anchos medios de 0,512 y 0,881 GWh. La menor puntuación combina precisión y amplitud; la cobertura nominal se verifica empíricamente. Véase la {ref}`evaluación común de incertidumbre <incertidumbre-comun>` para resultados de los 18 procedimientos.

## Contribución de la memoria y sensibilidad de la conclusión

Las ablaciones respaldan la actualización del banco y la corrección frente a M1 sin memoria, con mejoras de 7,87 % y 8,64 %. M4 mejora 4,17 % frente al control de capacidad igualada, aunque no acredita superar el margen del 3 %. No se demuestra una contribución favorable de la compuerta y el control por distancia numérica alcanza menor MASE. La ventaja frente al Top 10 no implica que todos los componentes sean necesarios.

La conclusión frente a los 17 comparadores se conserva con RMSE y los subconjuntos de primeras y últimas tres semillas. Frente a TimesNet, d_z=0,452 (IC95 % [0,374; 0,536]). La evaluación homogénea del checkpoint obtiene una brecha ajuste–parada de 14,15 %, con IC95 % [-2,52; 31,03] %. El {ref}`análisis de contribución <complementos-m4>` distingue evidencia de rendimiento, mecanismo y cambio temporal.

## Rendimiento y coste del pipeline

La {ref}`tabla maestra <tabla-maestra>` reúne 18 procedimientos bajo un protocolo común de coste. M4 obtiene el menor MASE de prueba (0,5770) y una latencia mediana de 12,98 ms por origen; TimesNet requiere 34,84 ms. La comparación incluye la preparación numérica y todos los componentes de cada pipeline. Los costes corresponden a realizaciones prefijadas y tres fechas; las métricas de precisión conservan las cinco o diez semillas de prueba. El tamaño de los parámetros se distingue de la memoria residente del proceso.